In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Nerovnost CHSH

*Odhadovaná spotřeba: dvě minuty na procesoru Heron r2 (POZNÁMKA: Jde pouze o odhad. Skutečná doba výpočtu se může lišit.)*
## Pozadí
V tomto tutoriálu spustíš experiment na kvantovém počítači a demonstruješ porušení nerovnosti CHSH pomocí primitivy Estimator.

Nerovnost CHSH, pojmenovaná po autorech Clauserovi, Hornovi, Shimonym a Holtovi, slouží k experimentálnímu důkazu Bellova teorému (1969). Tento teorém tvrdí, že teorie lokálních skrytých proměnných nemohou vysvětlit některé důsledky provázanosti v kvantové mechanice. Porušení nerovnosti CHSH se používá k prokázání toho, že kvantová mechanika je neslučitelná s teoriemi lokálních skrytých proměnných. Jde o důležitý experiment pro pochopení základů kvantové mechaniky.

Nobelova cena za fyziku za rok 2022 byla udělena Alainu Aspectovi, Johnu Clauserovi a Antonu Zeilingerovi mimo jiné za jejich průkopnickou práci v oblasti kvantové informační vědy, a zejména za jejich experimenty s provázanými fotony demonstrující porušení Bellových nerovností.
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

* Qiskit SDK v1.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 nebo novější
## Nastavení

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

## Krok 1: Mapování klasických vstupů na kvantový problém
Pro tento experiment vytvoříme provázaný pár, na němž změříme každý Qubit ve dvou různých bázích. Báze pro první Qubit označíme $A$ a $a$ a báze pro druhý Qubit $B$ a $b$. To nám umožní vypočítat veličinu CHSH $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Každá observabla nabývá hodnoty $+1$ nebo $-1$. Je zřejmé, že jeden z členů $B\pm b$ musí být $0$ a druhý $\pm 2$. Proto $S_1 = \pm 2$. Střední hodnota $S_1$ musí splňovat nerovnost:

$$
|\langle S_1 \rangle|\leq 2.
$$

Rozvinutím $S_1$ do členů $A$, $a$, $B$ a $b$ dostaneme:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Lze definovat další veličinu CHSH $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

Z toho plyne další nerovnost:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

Pokud by kvantová mechanika mohla být popsána teoriemi lokálních skrytých proměnných, předchozí nerovnosti by musely platit. Jak je však v tomto tutoriálu ukázáno, tyto nerovnosti mohou být na kvantovém počítači porušeny. Kvantová mechanika tedy není slučitelná s teoriemi lokálních skrytých proměnných.
Pokud se chceš dozvědět více o teorii, prozkoumej [Provázanost v praxi](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game) s Johnem Watrousem.
Vytvoříš provázaný pár mezi dvěma Qubity v kvantovém počítači pomocí Bellova stavu $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Pomocí primitivy Estimator můžeš přímo získat potřebné střední hodnoty ($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$ a $\langle ab \rangle$) pro výpočet středních hodnot obou veličin CHSH $\langle S_1\rangle$ a $\langle S_2\rangle$. Před zavedením primitivy Estimator by bylo nutné sestrojit střední hodnoty z výsledků měření.

Druhý Qubit změříš v bázích $Z$ a $X$. První Qubit bude také měřen v ortogonálních bázích, ale s úhlem vůči druhému Qubitu, který budeme přeléhat od $0$ do $2\pi$. Jak uvidíš, primitiva Estimator značně usnadňuje spouštění parametrizovaných obvodů. Místo vytváření série obvodů CHSH stačí vytvořit pouze *jeden* obvod CHSH s parametrem určujícím úhel měření a sérii hodnot fáze pro tento parametr.

Nakonec výsledky analyzuješ a vyneseš je do grafu v závislosti na úhlu měření. Uvidíš, že pro určitý rozsah úhlů měření platí $|\langle S_1\rangle| > 2$ nebo $|\langle S_2\rangle| > 2$, což demonstruje porušení nerovnosti CHSH.

In [2]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_kingston'

### Create a parameterized CHSH circuit

First, we write the circuit with the parameter $\theta$, which we call `theta`. The [`Estimator` primitive](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) can enormously simplify circuit building and output analysis by directly providing expectation values of observables. Many problems of interest, especially for near-term applications on noisy systems, can be formulated in terms of expectation values. `Estimator` (V2) primitive can automatically change measurement basis based on the supplied observable.

In [3]:
theta = Parameter("$\\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif" alt="Output of the previous code cell" />

### Vytvoření parametrizovaného obvodu CHSH
Nejprve zapíšeme obvod s parametrem $\theta$, který nazveme `theta`. [Primitiva `Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) může výrazně zjednodušit sestavování obvodů a analýzu výstupů tím, že přímo poskytuje střední hodnoty observabl. Mnoho zajímavých problémů, zejména pro krátkodobé aplikace na zašuměných systémech, lze formulovat pomocí středních hodnot. Primitiva `Estimator` (V2) umí automaticky měnit měřicí bázi na základě zadané observably.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as list of lists in order to work
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### Vytvoření seznamu hodnot fáze pro pozdější přiřazení
Po vytvoření parametrizovaného obvodu CHSH vytvoříš seznam hodnot fáze, které budou obvodu přiřazeny v dalším kroku. Pomocí následujícího kódu lze vytvořit seznam 21 hodnot fáze rovnoměrně rozložených od $0$ do $2 \pi$, tedy $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$.

In [5]:
# <CHSH1> = <AB> - <Ab> + <aB> + <ab> -> <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <CHSH2> = <AB> + <Ab> - <aB> + <ab> -> <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### Observably
Nyní potřebujeme observably, z nichž vypočítáme střední hodnoty. V našem případě se díváme na ortogonální báze pro každý Qubit a necháváme parametrizovanou $Y$-rotaci prvního Qubitu přelévat měřicí bázi téměř spojitě vůči bázi druhého Qubitu. Zvolíme proto observably $ZZ$, $ZX$, $XZ$ a $XX$.

In [6]:
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)

chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif" alt="Output of the previous code cell" />

## Krok 2: Optimalizace problému pro spuštění na kvantovém hardware

Aby se zkrátila celková doba provádění úloh, primitivy V2 přijímají pouze obvody a observably, které odpovídají instrukcím a propojení podporovanému cílovým systémem (označované jako obvody a observably ve formátu ISA — instruction set architecture).

### ISA Circuit

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### ISA Observables

Podobně je nutné transformovat observably, aby byly kompatibilní s backendem, než spustíme úlohy pomocí [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run). Transformaci provedeme metodou `apply_layout` objektu `SparsePauliOp`.

In [8]:
# To run on a local simulator:
# Use the StatevectorEstimator from qiskit.primitives instead.

estimator = Estimator(mode=backend)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA Observables
    individual_phases,  # Parameter values
)

job_result = estimator.run(pubs=[pub]).result()

## Krok 3: Spuštění pomocí primitiv Qiskit
Celý experiment spustíme jediným voláním [`Estimatoru`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2).
Pomocí primitivy [Qiskit Runtime `Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) vypočítáme střední hodnoty. Metoda `EstimatorV2.run()` přijímá iterovatelnou kolekci `primitive unified blocs (PUBs)`. Každý PUB je iterovatelná kolekce ve formátu `(circuit, observables, parameter_values: Optional, precision: Optional)`.

In [9]:
chsh1_est = job_result[0].data.evs[0]
chsh2_est = job_result[0].data.evs[1]

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))

# results from hardware
ax.plot(phases / np.pi, chsh1_est, "o-", label="CHSH1", zorder=3)
ax.plot(phases / np.pi, chsh2_est, "o-", label="CHSH2", zorder=3)

# classical bound +-2
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")

# quantum bound, +-2√2
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

# set x tick labels to the unit of pi
ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

# set labels, and legend
plt.xlabel("Theta")
plt.ylabel("CHSH witness")
plt.legend()
plt.show()

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/f6267448-0.avif" alt="Output of the previous code cell" />

In the figure, the lines and gray areas delimit the bounds; the outer-most (dash-dotted) lines delimit the quantum-bounds ($\pm 2$), whereas the inner (dashed) lines delimit the classical bounds ($\pm 2\sqrt{2}$). You can see that there are regions where the CHSH witness quantities exceeds the classical bounds. Congratulations! You have successfully demonstrated the violation of CHSH inequality in a real quantum system!

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3xxAgm1SF1wGp9k)